# Spectrogram Generation

In this notebook, the previously labeled vibration time-series segments are transformed into time–frequency representations using short-time Fourier transform (STFT). The resulting spectrograms are normalized and converted into RGB images by stacking the X, Y, and Z acceleration axes. These images serve as input for the subsequent convolutional neural network (CNN) used for chatter detection.

In [22]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import spectrogram
from PIL import Image
import os
from tqdm import tqdm
import random

# In-/ Output Structure & Class-Mapping

In [23]:
DATASETS_TO_PROCESS = ("turning", "cnc_machining")

DATASET_CONFIGS = {
    "turning": {
        "data_root": Path("../data/01_windowed_labeled_2,5s"),
        "output_root": Path("../data/02_spectrograms_150x100px_dataset"),
        "manifest_path": Path("../reports/manifests/turning_split_seed42.csv"),
        "sample_rate_hz": 50000,
        "max_freq_hz": 5000,
    },
    "cnc_machining": {
        "data_root": Path("../data/01_windowed_labeled_cnc_machining"),
        "output_root": Path("../data/02_spectrograms_150x100px_cnc_machining_dataset"),
        "manifest_path": Path("../reports/manifests/cnc_machining_split_seed42.csv"),
        "sample_rate_hz": 2000,
        "max_freq_hz": 1000,
    },
}

REPO_ROOT = Path("..").resolve()

CLASS_MAP = {
    "chatter": "chatter",
    "no_chatter": "no_chatter" 
}

SEED = 42
OVERLAP_FRACTION = 0.75
DB_PERCENTILES = (1, 99)
DB_RANGE_SAMPLE_LIMIT = 1000
EPS = 1e-12

# Frequency-axis pixels divided by time-axis pixels.
# 100 / 150 preserves the previous image proportions without forcing a fixed size.
FREQ_TIME_PIXEL_RATIO = 100 / 150


# Spectrogram configuration
- FREQ_TIME_PIXEL_RATIO: desired ratio of frequency pixels to time pixels
- nperseg: calculated per signal from the desired pixel ratio, signal length, overlap, sample rate, and max_freq_hz
larger values -> better frequency resolution, lower time resolution
- segment count M is solved from K(1-O)M^2 + K O M - N_total = 0, where K = V / (max_freq_hz / fs)
- for max_freq_hz = fs / 2, K = 2V and M = (-2VO + sqrt((2VO)^2 + 8V(1-O)N_total)) / (4V(1-O)); nperseg = round(KM)
- max_freq_hz: configured per dataset; limits analysis to physically available/relevant frequency range
- DB_PERCENTILES: spectrogram is converted to decibel scale and robustly scaled per dataset from training data
- DB_RANGE_SAMPLE_LIMIT: maximum number of training windows used to estimate the dataset-level dB range
- image size: comes from the STFT geometry; generated images are not resized in this notebook

In [24]:
for config in DATASET_CONFIGS.values():
    config["output_root"].mkdir(parents=True, exist_ok=True)


## Spectrogram generator function

In [25]:
def optimal_nperseg(total_samples, fs, max_freq_hz, freq_time_pixel_ratio, overlap_fraction=OVERLAP_FRACTION):
    """Calculate STFT segment length from the desired frequency/time pixel ratio."""
    total_samples = int(total_samples)
    fs = float(fs)
    max_freq_hz = min(float(max_freq_hz), fs / 2)
    freq_time_pixel_ratio = float(freq_time_pixel_ratio)

    if total_samples <= 0:
        raise ValueError("total_samples must be positive")
    if fs <= 0:
        raise ValueError("fs must be positive")
    if max_freq_hz <= 0:
        raise ValueError("max_freq_hz must be positive")
    if freq_time_pixel_ratio <= 0:
        raise ValueError("freq_time_pixel_ratio must be positive")
    if not 0 <= overlap_fraction < 1:
        raise ValueError("overlap_fraction must be in [0, 1)")

    visible_frequency_fraction = max_freq_hz / fs
    samples_per_time_bin = freq_time_pixel_ratio / visible_frequency_fraction
    a = samples_per_time_bin * (1 - overlap_fraction)
    b = samples_per_time_bin * overlap_fraction
    segment_count = (-b + np.sqrt(b**2 + 4 * a * total_samples)) / (2 * a)
    return max(1, min(total_samples, int(round(samples_per_time_bin * segment_count))))


def spectrogram_db(sig, fs, max_freq_hz):
    nperseg = optimal_nperseg(len(sig), fs, max_freq_hz, FREQ_TIME_PIXEL_RATIO)
    noverlap = int(OVERLAP_FRACTION * nperseg)
    max_freq_hz = min(float(max_freq_hz), fs / 2)

    f, t, Sxx = spectrogram(
        sig,
        fs=fs,
        nperseg=nperseg,
        noverlap=noverlap,
        mode="psd"
    )

    mask = f <= max_freq_hz
    Sxx = Sxx[mask, :]

    return 10 * np.log10(Sxx + EPS)


def create_spectrogram(sig, fs, max_freq_hz, db_range):
    db_min, db_max = db_range
    Sxx_db = spectrogram_db(sig, fs, max_freq_hz)
    Sxx_db = np.clip(Sxx_db, db_min, db_max)

    return 1.0 - (Sxx_db - db_min) / (db_max - db_min)

In [26]:
def repo_relative(path: Path) -> str:
    path = Path(path).resolve()
    return str(path.relative_to(REPO_ROOT)) if path.is_relative_to(REPO_ROOT) else str(path)

dataset_manifests = {}
split_paths = {}
required_columns = {"source_dataset", "npz_path", "split", "label"}

for dataset_name in DATASETS_TO_PROCESS:
    config = DATASET_CONFIGS[dataset_name]
    manifest = pd.read_csv(config["manifest_path"])
    manifest = manifest[manifest["source_dataset"] == dataset_name].copy()
    missing_columns = required_columns - set(manifest.columns)
    if missing_columns:
        raise ValueError(f"{dataset_name} manifest is missing required columns: {sorted(missing_columns)}")
    dataset_manifests[dataset_name] = manifest

    for split in sorted(manifest["split"].unique()):
        split_labels = sorted(manifest.loc[manifest["split"] == split, "label"].unique())
        for label in split_labels:
            rows = manifest[(manifest["split"] == split) & (manifest["label"] == label)]
            split_paths[(dataset_name, split, label)] = [REPO_ROOT / rel_path for rel_path in rows["npz_path"]]

for (dataset_name, split, label), paths in split_paths.items():
    print(f"{dataset_name}/{split}/{label}: {len(paths)}")


def estimate_dataset_db_range(dataset_name, manifest):
    config = DATASET_CONFIGS[dataset_name]
    train_rows = manifest[manifest["split"] == "train"]
    if train_rows.empty:
        raise ValueError(f"{dataset_name} has no training rows for dB range estimation")

    if len(train_rows) > DB_RANGE_SAMPLE_LIMIT:
        train_rows = train_rows.sample(n=DB_RANGE_SAMPLE_LIMIT, random_state=SEED)

    db_values = []
    for rel_path in tqdm(train_rows["npz_path"], desc=f"Estimating {dataset_name} dB range"):
        data = np.load(REPO_ROOT / rel_path)
        fs = float(data["fs"]) if "fs" in data.files else config["sample_rate_hz"]
        for axis in ("X", "Y", "Z"):
            db_values.append(spectrogram_db(data[axis], fs, config["max_freq_hz"]).ravel())

    db_values = np.concatenate(db_values)
    db_values = db_values[np.isfinite(db_values)]
    db_min, db_max = np.percentile(db_values, DB_PERCENTILES)
    if db_max <= db_min:
        raise ValueError(f"Invalid {dataset_name} dB range: {(db_min, db_max)}")
    return float(db_min), float(db_max)


DATASET_DB_RANGES = {
    dataset_name: estimate_dataset_db_range(dataset_name, manifest)
    for dataset_name, manifest in dataset_manifests.items()
}

for dataset_name, db_range in DATASET_DB_RANGES.items():
    print(f"{dataset_name} dB range ({DB_PERCENTILES[0]}-{DB_PERCENTILES[1]} percentile): {db_range}")


turning/test/chatter: 4
turning/test/no_chatter: 81
turning/train/no_chatter: 472
turning/validation/chatter: 2
turning/validation/no_chatter: 92
cnc_machining/test/anomaly: 35
cnc_machining/test/nominal: 245
cnc_machining/train/nominal: 1142
cnc_machining/validation/anomaly: 35
cnc_machining/validation/nominal: 245


Estimating turning dB range:   0%|          | 0/472 [00:00<?, ?it/s]

Estimating cnc_machining dB range: 100%|██████████| 1000/1000 [00:23<00:00, 42.81it/s]


turning dB range (1-99 percentile): (-119.32903202008087, -75.59818403048429)
cnc_machining dB range (1-99 percentile): (-71.6695327758789, -19.642833709716797)


# RGB Image Conversion
This function converts the labeled vibration time-series segment into a 3-channel spectrogram image.
Combines three spectrograms into a single 3-channel image:

R = X-axis
G = Y-axis
B = Z-axis

Each vibration segment is transformed into a three-channel spectrogram image, where each channel corresponds to a different sensor axis. This encoding preserves spatial vibration directionality while enabling convolutional neural networks to learn cross-axis correlations relevant for chatter detection.

**Advantages of these representation:**
- harmonic stripes (chatter)
- broadband noise (instability)
- axis coupling effects

In [27]:
def process_npz(dataset_name, npz_path, split_folder, target_label_folder):
    data = np.load(npz_path)
    config = DATASET_CONFIGS[dataset_name]

    x = data["X"]
    y = data["Y"]
    z = data["Z"]

    label = str(data["label"])
    # label_folder wird nicht mehr aus label abgeleitet, sondern kommt vom Caller
    fs = float(data["fs"]) if "fs" in data.files else config["sample_rate_hz"]
    db_range = DATASET_DB_RANGES[dataset_name]

    specs = [
        create_spectrogram(x, fs, config["max_freq_hz"], db_range),
        create_spectrogram(y, fs, config["max_freq_hz"], db_range),
        create_spectrogram(z, fs, config["max_freq_hz"], db_range)
    ]

    rgb = np.stack(specs, axis=-1)
    rgb = np.clip(rgb, 0, 1)
    rgb = np.flipud(rgb)

    img = Image.fromarray((rgb * 255).astype(np.uint8))

    experiment = npz_path.parent.parent.name
    out_name = f"{dataset_name}_{experiment}_{npz_path.stem}_{label}.png"

    out_path = config["output_root"] / split_folder / target_label_folder / out_name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    img.save(out_path)
    return out_path

In [28]:
generated_rows = []

for (dataset_name, split_folder, label_folder), npz_files in split_paths.items():
    desc = f"Processing {dataset_name}/{split_folder}/{label_folder}"
    for npz_file in tqdm(npz_files, desc=desc):
        try:
            out_path = process_npz(dataset_name, npz_file, split_folder, label_folder)
            generated_rows.append({
                "source_dataset": dataset_name,
                "npz_path": repo_relative(npz_file),
                "image_path": repo_relative(out_path),
            })
        except Exception as e:
            print(f"Error in {npz_file}: {e}")

generated_images = pd.DataFrame(generated_rows)
if not generated_images.empty:
    for dataset_name, manifest in dataset_manifests.items():
        config = DATASET_CONFIGS[dataset_name]
        dataset_images = generated_images[generated_images["source_dataset"] == dataset_name]
        updated = manifest.drop(columns=["image_path"], errors="ignore").merge(
            dataset_images[["npz_path", "image_path"]], on="npz_path", how="left"
        )
        updated.to_csv(config["manifest_path"], index=False)
        print(f"Wrote image paths to {config['manifest_path']}")


Processing cnc_machining/validation/nominal: 100%|██████████| 245/245 [00:14<00:00, 16.80it/s]


Wrote image paths to ../reports/manifests/turning_split_seed42.csv
Wrote image paths to ../reports/manifests/cnc_machining_split_seed42.csv
